# Neural Operators — Hands-on Lab (step by step)

Companion notebook to **Part 3 (Neural Operators)**.

This is the *step-by-step* version: every code cell is at most ten lines and
does exactly one thing, so you can run it, read it, change it, and run it
again. If you would rather see the whole thing as one flowing demo, use
`neural_operators_handson.ipynb` instead.

| # | Experiment | What you build |
|---|------------|----------------|
| 1 | **The data** | a solver that maps a medium $a$ to a pressure field $u$ |
| 2 | **An operator from scratch** | a spectral layer, then the resolution test |
| 3 | **The library FNO** | the same thing, production-grade, plus its knobs |
| 4 | **Mini-FourCastNet** | a six-hour weather step operator, rolled out |

**How to use it.** Run the cells in order. Wherever you see **Try it**, change
a number and re-run — that is the point of the lab.

**One convention:** plumbing lives in `nolab.py` next to this notebook (the
finite-difference solver, plotting boilerplate, the weather download). The
things this lab is *about* — the spectral layer, the model, the training
loop — are written out in full here, where you can edit them.

In [ ]:
# Fresh environment / Colab: run once, then restart the kernel if asked.
# %pip install -q numpy scipy matplotlib torch neuraloperator xarray zarr gcsfs dask

## Setup

Two cells: the imports, then the device. Nothing interesting happens yet.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
rng = np.random.default_rng(0)

`pick_device()` returns `"cuda"` if you have an NVIDIA GPU, `"mps"` on Apple
Silicon, and `"cpu"` otherwise.

The Apple-Silicon check is not just "is MPS available?" — Experiment 2 needs
**complex-valued weights** and **2-D FFTs**, and older PyTorch builds cannot
do those on Metal. So the helper actually runs a miniature spectral layer
first and only returns `"mps"` if it survives.

In [ ]:
from nolab import pick_device, grf, solve_darcy, two_phase, show

device = pick_device()
print("running on:", device)

`FAST = True` keeps everything laptop-sized. Set it to `False` once you have
seen the whole notebook run and want better numbers.

In [ ]:
FAST    = True                      # False -> more data, longer training
N_TRAIN = 300 if FAST else 1000     # training media
N_TEST  = 32  if FAST else 100      # test media, solved at 3 resolutions each
EPOCHS  = 15  if FAST else 60

---
# Experiment 1 — the data

Steady flow through a porous medium (groundwater through soil, oil through
rock). The pressure $u$ inside the unit square obeys

$$-\nabla\cdot\big(a(x)\,\nabla u(x)\big) = f(x),
\qquad u = 0 \text{ on the boundary.}$$

- $a(x)$ — the **permeability**: how easily fluid moves at each point. This
  is the *input*.
- $f(x)$ — a source term (we keep it constant).
- $u(x)$ — the resulting **pressure**. This is the *output*.

A classical solver takes one $a$ and returns one $u$. Our goal for the rest
of the notebook is to learn the map $G: a \mapsto u$ **once**, for the whole
family of media.

### One sample

`two_phase` draws a smooth random field and thresholds it into two materials:
high permeability (`a = 12`, think sand) and low (`a = 3`, think clay).
`solve_darcy` is the finite-difference ground truth.

In [ ]:
a = two_phase(64, rng)     # permeability field: 12 in the sand, 3 in the clay
u = solve_darcy(a)         # pressure field, from the finite-difference solver

print("a:", a.shape, a.dtype, "| values:", np.unique(a))
print("u:", u.shape, "| range: %.4f .. %.4f" % (u.min(), u.max()))

In [ ]:
show({"input  $a(x)$": a, "output  $u(x)$": u}, cmaps=["viridis", "inferno"])

Look at the two pictures together. The pressure is high where the source
pushes fluid in and drops to zero at the boundary — but it does **not** drop
smoothly: it bends around the low-permeability blobs. The output field
inherits the *geometry* of the input field.

That relationship is the operator. Everything below is about learning it.

### Try it: the correlation length

`tau` controls how fine-grained the medium is. Larger `tau` → smaller blobs →
more small-scale structure in the pressure. Those small features are the
**high frequencies**, and they are exactly what the FNO will have to work for
later.

In [ ]:
for tau in (1.5, 3.0, 6.0):
    a_t = two_phase(64, np.random.default_rng(0), tau=tau)
    show({f"$a$  (tau = {tau})": a_t, "$u$": solve_darcy(a_t)},
         cmaps=["viridis", "inferno"], size=2.7)

### Try it: the forcing

Flip the sign of $f$ and the whole pressure field flips with it: the operator
is *linear in $f$*. It is emphatically **not** linear in $a$ — that is what
makes learning $G$ a real problem.

In [ ]:
a0 = two_phase(64, np.random.default_rng(1))

show({"$f = +1$": solve_darcy(a0, f_val=1.0),
      "$f = -1$": solve_darcy(a0, f_val=-1.0)})

### Try it: draw your own medium

Edit the string below — `#` is a high-permeability channel, `.` is clay.
Keep it small; it gets resampled to a 64×64 grid.

Worth drawing: a winding channel from one edge to the other, or a **dam with
a small gap** (the whole pressure drop concentrates in the gap). We will feed
your drawing to the trained model in Experiment 3 as an out-of-distribution
test, so make it weird.

In [ ]:
from nolab import ascii_medium

ART = """
..###########.
............#.
..###########.
..#...........
..###########.
"""
a_art = ascii_medium(ART)

In [ ]:
u_art = solve_darcy(a_art)

show({"your medium  $a$": a_art, "pressure  $u$": u_art},
     cmaps=["viridis", "inferno"])

### From a solver to a dataset

Every new medium is one more example of the map $a \mapsto u$. One decision
here matters more than it looks.

We solve **once, at 128×128** — the finest grid we can afford — and then
*sample* that single solution onto coarser grids. We do **not** solve
separately on each grid.

Why it matters: a coarse solve carries its own discretisation error (about
**7%** at 32×32 and **2%** at 64×64 for this problem). If we used coarse
solves as ground truth, the error-versus-resolution curve in Experiment 2
would be measuring the *solver's* error as much as the model's. Sampling one
high-fidelity solution keeps the target function fixed and changes only how
densely we look at it — which is precisely the question operator learning
asks.

In [ ]:
def hi_fi(r):
    """One medium and its pressure field, computed once at 128x128."""
    a = np.where(grf(128, r) >= 0, 12.0, 3.0).astype(np.float32)
    return a, solve_darcy(a)

def at_res(a, u, m):
    """Sample that same medium and that same solution on an m x m grid."""
    s = 128 // m
    return a[::s, ::s], u[::s, ::s]

The **training set** is that construction, kept at 32×32 — deliberately
coarse, because the whole point is to train cheap and evaluate fine.

In [ ]:
train_a, train_u = [], []
for _ in range(N_TRAIN):
    a_hi, u_hi = hi_fi(rng)
    a32, u32 = at_res(a_hi, u_hi, 32)
    train_a.append(a32); train_u.append(u32)
train_a, train_u = np.stack(train_a), np.stack(train_u)

print(train_a.shape, "->", train_u.shape)

In [ ]:
from nolab import show_pairs

show_pairs(train_a, train_u)

The **test triplets** use the same generator, but keep all three samplings of
each solution. Same medium, same underlying pressure field — three grids.

In [ ]:
test = {m: ([], []) for m in (32, 64, 128)}
for _ in range(N_TEST):
    a_hi, u_hi = hi_fi(rng)
    for m in (32, 64, 128):
        a_m, u_m = at_res(a_hi, u_hi, m)
        test[m][0].append(a_m); test[m][1].append(u_m)
test = {m: (np.stack(v[0]), np.stack(v[1])) for m, v in test.items()}

print({m: test[m][0].shape for m in test})

### Normalisation

Standard practice: rescale with **training** statistics only, so the test
sets get no information they should not have. Note the shape convention
PyTorch wants — `(batch, channels, height, width)` — hence the `unsqueeze(1)`.

In [ ]:
A_MU, A_SD, U_SD = train_a.mean(), train_a.std(), train_u.std()

def to_torch(a_np, u_np):
    x = torch.from_numpy((a_np - A_MU) / A_SD).unsqueeze(1).float()
    y = torch.from_numpy(u_np / U_SD).unsqueeze(1).float()
    return x, y

x_train, y_train = to_torch(train_a, train_u)
x_test = {m: to_torch(*test[m])[0] for m in test}
y_test = {m: to_torch(*test[m])[1] for m in test}

In [ ]:
loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)

print(x_train.shape, "->", y_train.shape, "|", len(loader), "batches")

---
# Experiment 2 — a neural operator from scratch

A normal network layer multiplies by a matrix. An operator layer integrates
against a **kernel**:

$$(\mathcal{K}v)(x) = \int_\Omega \kappa(x-y)\,v(y)\,\mathrm{d}y$$

By the convolution theorem that integral is a plain multiplication in Fourier
space — one complex **gain per frequency**. So instead of storing a kernel we
store the gains and do

$$v \;\longmapsto\; \mathcal{F}^{-1}\big(R \cdot \mathcal{F}v\big),$$

keeping only the lowest $k$ frequencies and setting the rest to zero.

The payoff: $R$ is indexed by *frequency*, which belongs to the domain — not
by *grid point*, which belongs to the sampling. Same weights, any grid.

### The spectral layer

Ten lines. `rfft2` goes to Fourier space, the `einsum` applies one learned
$C\times C$ complex matrix per kept mode, `irfft2` comes back. Everything
above mode `k` stays zero — that is the truncation.

In [ ]:
class Spectral2d(nn.Module):
    def __init__(self, c, k):
        super().__init__(); self.k = k
        self.R = nn.Parameter(0.02 * torch.randn(c, c, k, k, dtype=torch.cfloat))
    def forward(self, v):
        vh = torch.fft.rfft2(v)                          # F
        out = torch.zeros_like(vh)
        out[..., :self.k, :self.k] = torch.einsum(       # R . (Fv)
            "bixy,ioxy->boxy", vh[..., :self.k, :self.k], self.R)
        return torch.fft.irfft2(out, s=v.shape[-2:])     # F^-1

### What does that layer actually do?

Before training anything, push a medium through an **untrained** spectral
layer. It throws away every frequency above $k$, so whatever comes out is
necessarily smooth. A spectral layer is a learnable low-pass filter.

In [ ]:
layer = Spectral2d(c=1, k=6)
with torch.no_grad():
    smoothed = layer(torch.from_numpy(a)[None, None].float())

show({"input $a$": a, "after one untrained spectral layer": smoothed[0, 0]},
     cmaps="viridis")

### The block

The spectral path is global but blurry — it dropped the high frequencies. So
each block adds a **pointwise** $1\times1$ convolution beside it, which is
local and sharp, and a nonlinearity:

$$v_{\ell+1} = \sigma\big(W v_\ell + \mathcal{K} v_\ell\big)$$

In [ ]:
class Block(nn.Module):
    def __init__(self, c, k):
        super().__init__()
        self.S = Spectral2d(c, k)          # global, smooth
        self.W = nn.Conv2d(c, c, 1)        # local, sharp
    def forward(self, v):
        return F.gelu(self.S(v) + self.W(v))

### One more ingredient: coordinates

The boundary pins $u = 0$, so the answer depends on **where** you are, not
only on the medium around you. But spectral and $1\times1$ layers are
*translation-equivariant* — slide the input, the output slides with it — so
a stack of them literally cannot represent "…and it is zero at the edge".

The fix is to hand the network two extra channels containing the $x$ and $y$
coordinates. Without them this model gets stuck around 0.5 relative error no
matter how long you train it. (You will get to check that claim at the end of
this experiment.)

In [ ]:
class WithGrid(nn.Module):
    """(a) -> (a, x, y): append two coordinate channels."""
    def forward(self, v):
        b, _, h, w = v.shape
        gy, gx = torch.meshgrid(torch.linspace(0, 1, h),
                                torch.linspace(0, 1, w), indexing="ij")
        g = torch.stack([gx, gy]).to(v).expand(b, -1, -1, -1)
        return torch.cat([v, g], 1)

### Assemble

Lift to `c` channels, apply `L` blocks, project back to one channel. Lift and
projection are $1\times1$ convolutions: pointwise, so they work at any
resolution.

Read the parameter count carefully — the grid size appears nowhere in it.

In [ ]:
def build_no(c=32, k=12, L=4):
    return nn.Sequential(WithGrid(),                    # (a, x, y)
                         nn.Conv2d(3, c, 1),            # lift P
                         *[Block(c, k) for _ in range(L)],
                         nn.Conv2d(c, 1, 1))            # project Q

model = build_no().to(device)
n_par = sum(p.numel() for p in model.parameters())
print(f"{n_par:,} parameters -- and not one of them mentions the grid size")

### The loss

Relative $L^2$ error: the error norm divided by the norm of the truth. Using
the *relative* error matters here — it is comparable across resolutions,
where an absolute error would not be.

In [ ]:
def rel_l2(pred, true):
    """Relative L2 error per sample, averaged over the batch."""
    err = (pred - true).flatten(1).norm(dim=1)
    return (err / true.flatten(1).norm(dim=1)).mean()

### The training loop

Nothing operator-specific here — it is the ordinary PyTorch loop. Worth
noticing: the model never sees anything but 32×32 data.

In [ ]:
def train(model, loader, epochs, lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = rel_l2(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            hist.append(loss.item())
    return hist

In [ ]:
hist = train(model, loader, EPOCHS)

plt.semilogy(hist); plt.xlabel("step"); plt.ylabel("relative $L^2$")
plt.title("training loss -- 32x32 data only"); plt.show()

### The experiment: train coarse, test fine

The model has only ever seen 32×32 grids. Now hand it the **same media**
sampled at 64×64 and 128×128 — grids it has never seen, with 4× and 16× as
many points. No retraining, no interpolation of weights.

In [ ]:
@torch.no_grad()
def evaluate(m):
    """Relative L2 on the test set, at each resolution."""
    return {r: round(float(rel_l2(m(x_test[r].to(device)).cpu(), y_test[r])), 4)
            for r in (32, 64, 128)}

err_no = evaluate(model)
print("neural operator:", err_no)

In [ ]:
show({f"prediction @ {r}x{r}":
      model(x_test[r][:1].to(device)).detach().cpu()[0, 0] for r in (32, 64, 128)},
     suptitle="the same weights, on three different grids")

The three numbers should be close. They do creep up a little, and that part
is real rather than a bug: the 128x128 reference contains detail finer than
anything the 32x32 training samples could resolve, and no model invents what
it was never shown. What matters is the *shape* of the curve — nearly flat.
Now compare it with something that is not an operator.

### The counter-experiment: a CNN

A fully-convolutional network *also* runs at any resolution — so is it also
an operator? No, and here is why: its $3\times3$ stencil spans $3/32$ of the
domain at training resolution but only $3/128$ at test resolution. The
**physical** reach of a layer shrinks as the grid refines, so the learned
rule quietly changes meaning.

To keep this honest the CNN gets the *same* inputs (including coordinates)
and a matched parameter budget. The only difference is local vs. global.

In [ ]:
def build_cnn(c=105, L=8):
    layers = [WithGrid(), nn.Conv2d(3, c, 3, padding=1)]
    for _ in range(L - 2):
        layers += [nn.GELU(), nn.Conv2d(c, c, 3, padding=1)]
    layers += [nn.GELU(), nn.Conv2d(c, 1, 3, padding=1)]
    return nn.Sequential(*layers)

cnn = build_cnn().to(device)
print(f"CNN: {sum(p.numel() for p in cnn.parameters()):,} parameters "
      f"(operator: {n_par:,})")

In [ ]:
train(cnn, loader, EPOCHS)
err_cnn = evaluate(cnn)

print("CNN:            ", err_cnn)
print("neural operator:", err_no)

In [ ]:
from nolab import bar_compare

bar_compare({"neural operator": [err_no[r] for r in (32, 64, 128)],
             "CNN": [err_cnn[r] for r in (32, 64, 128)]},
            ["32x32", "64x64", "128x128"],
            "relative $L^2$ error", "resolution transfer")

**What you should see.** The operator's bars are nearly level; the CNN's grow
steeply. Both models are fine at the resolution they were trained on — it is
what happens *away* from it that separates them. Note this is not a capacity
argument: the two have the same parameter budget and the same inputs.

**Try it (Experiment 2):**

- Delete `WithGrid()` from `build_no` and retrain. The error jumps to ≈0.5 at
  *every* resolution, and training longer makes it worse, not better. That is
  the translation-equivariance argument above, in numbers.
- Set `k = 4` in `build_no` and retrain. Where does the prediction go wrong
  first? (Look along the interfaces between the two materials.)
- Halve `L`. How much accuracy does depth actually buy?
- Evaluate the CNN at 128×128 after first downsampling its input to 32×32.
  Can you rescue it? What does that tell you about what it learned?

---
# Experiment 3 — the library FNO

Same architecture, production-grade, from `neuraloperator`. Two knobs matter
most:

- **`n_modes`** — the truncation $k$ per dimension. Parameters grow like
  $k^2c^2$ per layer, and $k$ can never usefully exceed the Nyquist limit of
  the *training* grid (16 for our 32×32 data).
- **`hidden_channels`** — the width $c$.

One thing the library does for you: it appends the coordinate channels
itself (`positional_embedding="grid"` by default). That is the same
`WithGrid` trick you wrote by hand — which is why `in_channels=1` here.

In [ ]:
from neuralop.models import FNO

fno = FNO(n_modes=(16, 16), hidden_channels=64, n_layers=4,
          in_channels=1, out_channels=1).to(device)

print(f"library FNO: {sum(p.numel() for p in fno.parameters()):,} parameters")

In [ ]:
train(fno, loader, EPOCHS)
err_fno = evaluate(fno)

print("library FNO:", err_fno)

### The truncation trade-off

Sweep $k$ and watch two things move in opposite directions: accuracy improves
with more modes, and so does the parameter count. Somewhere in between is the
model you actually want.

In [ ]:
sweep = {}
for k in (2, 4, 8, 16):
    m_ = FNO(n_modes=(k, k), hidden_channels=32, n_layers=4,
             in_channels=1, out_channels=1).to(device)
    train(m_, loader, max(EPOCHS // 2, 6))
    sweep[k] = (evaluate(m_)[64], sum(p.numel() for p in m_.parameters()))

print({k: v for k, v in sweep.items()})

In [ ]:
ks = list(sweep)
fig, ax1 = plt.subplots(figsize=(5.8, 3.4))
ax1.plot(ks, [sweep[k][0] for k in ks], "o-", color="#1F7FA8")
ax1.set_xlabel("kept modes $k$"); ax1.set_xscale("log", base=2)
ax1.set_ylabel("error @ 64x64", color="#1F7FA8")
ax2 = ax1.twinx(); ax2.plot(ks, [sweep[k][1] for k in ks], "s--", color="#C3423F")
ax2.set_ylabel("parameters", color="#C3423F")
plt.title("truncation: accuracy vs. size"); plt.show()

### Look at the fields, not only the loss

A single loss number hides *where* a model is wrong. For operator models the
routine diagnostic is to plot the input, the truth, the prediction and the
error side by side.

In [ ]:
i, r = 3, 64
pred = fno(x_test[r][i:i + 1].to(device)).detach().cpu()[0, 0]
truth = y_test[r][i, 0]

show({"$a$": x_test[r][i, 0], "$u$ true": truth, "$u$ predicted": pred,
      "$|\\hat u - u|$": (pred - truth).abs()},
     cmaps=["viridis", "inferno", "inferno", "magma"])

The error is not spread evenly — it clusters along the **interfaces** between
the two materials, where the pressure field kinks. Those kinks are exactly
the high frequencies the truncation threw away.

A sharper way to see it: take the Fourier transform of the error itself.

In [ ]:
E = np.abs(np.fft.fftshift(np.fft.fft2((pred - truth).numpy())))

plt.imshow(np.log10(E + 1e-9), cmap="magma"); plt.axis("off")
plt.title("error spectrum  $\\log_{10}|\\mathcal{F}(\\hat u - u)|$")
plt.colorbar(shrink=0.8); plt.show()

### The acid test: your own drawing

The model has only ever seen *random* blobby media. Your ASCII drawing from
Experiment 1 — straight channels, sharp corners, right angles — looks like
nothing in the training set. You built an out-of-distribution test by hand.

If the error is still moderate, the model learned something about the
**physics**, not just about the training distribution.

In [ ]:
x_d = torch.from_numpy((a_art - A_MU) / A_SD)[None, None].float()
u_hat = fno(x_d.to(device)).detach().cpu()[0, 0].numpy() * U_SD
rel = np.linalg.norm(u_hat - u_art) / np.linalg.norm(u_art)

show({"your $a$": a_art, "solver $u$": u_art, "FNO $\\hat u$": u_hat,
      f"error (rel. {rel:.1%})": np.abs(u_hat - u_art)},
     cmaps=["viridis", "inferno", "inferno", "magma"])

**Try it (Experiment 3):**

- Go back, draw something meaner (a maze, a dam with one narrow gap), re-run
  the two cells. Where does the model break down?
- With $k = 2$ the prediction is visibly blurred — why must it be? What can a
  two-mode function represent at all?
- Double `hidden_channels` at $k = 8$: does width substitute for modes?
- Time one forward pass at $32^2$, $64^2$, $128^2$. Does it scale like
  $N\log N$?

---
# Experiment 4 — a very small FourCastNet

Nothing forced the input to be a *coefficient*. It can be any function the
solution depends on — for instance the **current state** of a time-dependent
system. Then the operator is a time-stepper:

$$G_{6h}:\;\big(Z_{500},\,T_{850}\big)_t \;\longmapsto\;
\big(Z_{500},\,T_{850}\big)_{t+6h}$$

That is FourCastNet, at about 1/1000 scale: geopotential and temperature on a
$5.625^\circ$ grid — the whole planet in $32\times64 = 2048$ pixels.

The data is ERA5, regridded by WeatherBench 2 and streamed from their public
bucket. **It usually takes one to three minutes** — start the next cell and
read on. If it stalls, the helper hands back a clearly-labelled synthetic
stand-in instead, and `load_weather(timeout_s=0)` skips the download entirely.

In [ ]:
from nolab import load_weather

X, real = load_weather()        # synthetic stand-in if offline or slow

print(X.shape, "(time, channel, lat, lon) | real ERA5:", real)

In [ ]:
Xn = (X - X.mean((0, 2, 3), keepdims=True)) / X.std((0, 2, 3), keepdims=True)
x_now, x_next = torch.from_numpy(Xn[:-1]), torch.from_numpy(Xn[1:])
n_tr = int(0.8 * len(x_now))
wloader = DataLoader(TensorDataset(x_now[:n_tr], x_next[:n_tr]),
                     batch_size=16, shuffle=True)

print(len(x_now), "six-hour steps |", n_tr, "for training")

In [ ]:
show({"channel 0  (state at $t$)": Xn[0, 0], "channel 1": Xn[0, 1]},
     cmaps="RdBu_r")

### Train the six-hour step

Two input channels, two output channels — otherwise this is the same FNO you
already used. The training pairs are just consecutive snapshots.

In [ ]:
fcn = FNO(n_modes=(8, 16), hidden_channels=48, n_layers=4,
          in_channels=2, out_channels=2).to(device)

train(fcn, wloader, 3 if FAST else 15, lr=1e-3)
print("trained a 6-hour step operator")

### Roll it out

Apply the learned six-hour operator to its **own output**, twenty times, and
you have a five-day forecast.

The honest baseline is **persistence** — "tomorrow will be like today",
$\hat x_{t+k} = x_t$ — which is embarrassingly hard to beat at short lead
times. A model with any skill has to stay below that dashed line for a while.

In [ ]:
STEPS = 20                                  # 20 x 6 h = 5 days
start = n_tr + 10                           # forecast from unseen data
truth = x_next[start:start + STEPS]
x0 = x_now[start:start + 1]

preds, xk = [], x0.to(device)
with torch.no_grad():
    for _ in range(STEPS):
        xk = fcn(xk)
        preds.append(xk.cpu())

In [ ]:
rmse_m = [float((preds[s] - truth[s:s+1]).pow(2).mean().sqrt()) for s in range(STEPS)]
rmse_p = [float((x0 - truth[s:s+1]).pow(2).mean().sqrt()) for s in range(STEPS)]
lead = 6 * (np.arange(STEPS) + 1) / 24

plt.plot(lead, rmse_m, "o-", color="#1F7FA8", label="mini-FourCastNet")
plt.plot(lead, rmse_p, "s--", color="#888", label="persistence")
plt.xlabel("lead time [days]"); plt.ylabel("RMSE (normalised)")
plt.legend(); plt.title("forecast skill vs. doing nothing"); plt.show()

In [ ]:
show({"forecast +24 h": preds[3][0, 0], "truth +24 h": truth[3, 0],
      "forecast +120 h": preds[-1][0, 0], "truth +120 h": truth[-1, 0]},
     cmaps="RdBu_r")

**Try it (Experiment 4):**

- Where does the rollout drift first — the large scales or the small ones?
  Compare the two forecast panels, then check with the error-spectrum trick
  from Experiment 3.
- Train for more epochs. How far out can you push the crossing point with
  persistence?
- Add channels (`u10`/`v10` winds, `2m_temperature` from the same store).
  Does $Z_{500}$ improve?
- The FFT treats the map as periodic north–south, which the sphere is not.
  Swapping in spherical harmonics (`torch-harmonics`) is the SFNO idea —
  compare the behaviour near the poles.

---
## Where to go next

- `neuraloperator` docs and examples: <https://neuraloperator.github.io>
- WeatherBench 2: <https://sites.research.google/weatherbench/>
- The papers behind this lab: FNO (Li et al., 2021) · neural operators
  (Kovachki et al., 2023) · FourCastNet (Pathak et al., 2022) · SFNO
  (Bonev et al., 2023) · nested FNO for CO$_2$ storage (Wen et al., 2023).

*Part 3 — Neural Operators. Step-by-step edition.*